In [1]:
#Problem statement 1 School data
import sqlite3
conn=sqlite3.connect("school_db.sqlite")
cursor=conn.cursor()
cursor.execute("""
CREATE TABLE students(
student_id INTEGER,
student_name TEXT,
class TEXT,
marks INTEGER
)
""")
cursor.executemany(
    "INSERT INTO students VALUES (?,?,?,?)",
    [
        (1,"Arun","10A",85),
        (2,"Divya","10A",90),
        (3,"Kiran","10B",78),
        (4,"Meena","10B",88),
        (5,"Ravi","10A",92)
    ]
)
conn.commit()
cursor.execute("""
SELECT
COUNT(*) AS total_students,
AVG(marks) AS average_marks,
MAX(marks) AS highest_marks,
MIN(marks) AS lowest_marks
FROM students
""")
print(cursor.fetchone())
conn.close()

(5, 86.6, 92, 78)


In [2]:
#PROBLEM 2 — TWISTED (GROUP BY + HAVING)
import sqlite3

conn = sqlite3.connect("company_db.sqlite")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE employees (
    emp_id INTEGER,
    emp_name TEXT,
    department TEXT,
    salary INTEGER
)
""")

cursor.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?)",
    [
        (1, "Ravi", "IT", 50000),
        (2, "Sneha", "HR", 35000),
        (3, "Manoj", "IT", 45000),
        (4, "Priya", "Finance", 42000),
        (5, "Karthik", "Finance", 38000),
        (6, "Anu", "IT", 48000),
        (7, "Vijay", "HR", 30000)
    ]
)
conn.commit()

cursor.execute("""
SELECT department, AVG(salary) AS avg_salary
FROM employees
GROUP BY department
HAVING AVG(salary) > 40000
""")

for row in cursor.fetchall():
    print(row)

conn.close()


('IT', 47666.666666666664)


In [3]:
 #PROBLEM 3 — TWISTED (Multi-Aggregate + Business Logic)
import sqlite3

conn = sqlite3.connect("sales_db.sqlite")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER,
    category TEXT,
    quantity INTEGER,
    price_per_unit INTEGER
)
""")

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?)",
    [
        (1, "Electronics", 5, 15000),
        (2, "Electronics", 3, 12000),
        (3, "Furniture", 2, 25000),
        (4, "Furniture", 1, 30000),
        (5, "Clothing", 20, 2000),
        (6, "Clothing", 15, 1800),
        (7, "Electronics", 2, 18000),
        (8, "Furniture", 1, 15000)
    ]
)
conn.commit()

cursor.execute("""
SELECT
    category,
    SUM(quantity) AS total_quantity,
    SUM(quantity * price_per_unit) AS total_revenue
FROM orders
GROUP BY category
HAVING total_revenue > 50000
ORDER BY total_revenue DESC
""")

for row in cursor.fetchall():
    print(row)

conn.close()


('Electronics', 10, 147000)
('Furniture', 4, 95000)
('Clothing', 35, 67000)


In [4]:
# PROBLEM STATEMENT 4 — Academic Performance Analytics
# STEP 1: IMPORT LIBRARIES
import sqlite3
import pandas as pd

# STEP 2: CREATE DATABASE CONNECTION
conn = sqlite3.connect("academic_performance.db")
cursor = conn.cursor()

# STEP 3: CREATE TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS student_results (
    student_id INTEGER PRIMARY KEY,
    department TEXT NOT NULL,
    semester INTEGER NOT NULL,
    marks REAL NOT NULL CHECK (marks BETWEEN 0 AND 100)
)
""")

# STEP 4: INSERT STUDENT DATA
students = [
    (1, 'Computer Science', 1, 85),
    (2, 'Computer Science', 1, 75),
    (3, 'Computer Science', 2, 55),
    (4, 'Computer Science', 2, 45),
    (5, 'Computer Science', 3, 60),
    (6, 'Commerce', 1, 50),
    (7, 'Commerce', 1, 55),
    (8, 'Commerce', 2, 40),
    (9, 'Commerce', 2, 65),
    (10, 'Commerce', 3, 45),
    (11, 'Mathematics', 1, 90),
    (12, 'Mathematics', 1, 80),
    (13, 'Mathematics', 2, 85),
    (14, 'Mathematics', 2, 75),
    (15, 'Mathematics', 3, 95),
    (16, 'Physics', 1, 55),
    (17, 'Physics', 1, 60),
    (18, 'Physics', 2, 50),
    (19, 'Physics', 2, 65),
    (20, 'Physics', 3, 45)
]

cursor.executemany("""
INSERT OR REPLACE INTO student_results
(student_id, department, semester, marks)
VALUES (?, ?, ?, ?)
""", students)

conn.commit()

# STEP 5: DISPLAY ALL STUDENT RESULTS
print("ALL STUDENT RESULTS")

display(pd.read_sql_query("""
SELECT * FROM student_results
ORDER BY student_id
""", conn))

# STEP 6: DEPARTMENT-WISE AVERAGE MARKS
print("1. DEPARTMENT-WISE AVERAGE MARKS")

display(pd.read_sql_query("""
SELECT
    department,
    ROUND(AVG(marks), 2) AS average_marks
FROM student_results
GROUP BY department
""", conn))

# STEP 7: DEPARTMENTS WITH AVERAGE MARKS BELOW 60
print("2. DEPARTMENTS WITH AVERAGE BELOW 60")

display(pd.read_sql_query("""
SELECT
    department,
    ROUND(AVG(marks), 2) AS average_marks
FROM student_results
GROUP BY department
HAVING AVG(marks) < 60
""", conn))

# STEP 8: MAXIMUM AND MINIMUM MARKS
print("3. MAXIMUM AND MINIMUM MARKS BY DEPARTMENT")

display(pd.read_sql_query("""
SELECT
    department,
    MAX(marks) AS maximum_marks,
    MIN(marks) AS minimum_marks
FROM student_results
GROUP BY department
""", conn))

# STEP 9: DEPARTMENTS WITH AT LEAST 5 STUDENTS
print("4. DEPARTMENTS WITH AT LEAST 5 STUDENTS")

display(pd.read_sql_query("""
SELECT
    department,
    COUNT(student_id) AS total_students,
    ROUND(AVG(marks), 2) AS average_marks,
    MAX(marks) AS maximum_marks,
    MIN(marks) AS minimum_marks
FROM student_results
GROUP BY department
HAVING COUNT(student_id) >= 5
""", conn))

# STEP 10: COMPLETE DEPARTMENT PERFORMANCE REPORT
print("COMPLETE ACADEMIC PERFORMANCE REPORT")

display(pd.read_sql_query("""
SELECT
    department,
    COUNT(student_id) AS total_students,
    ROUND(AVG(marks), 2) AS average_marks,
    MAX(marks) AS maximum_marks,
    MIN(marks) AS minimum_marks,
    CASE
        WHEN AVG(marks) < 60 THEN 'Underperforming'
        ELSE 'Satisfactory'
    END AS performance_status
FROM student_results
GROUP BY department
ORDER BY average_marks DESC
""", conn))

# STEP 11: CLOSE DATABASE
conn.close()

print("Academic Performance Analytics Completed!")



ALL STUDENT RESULTS


,student_id,department,semester,marks
0,1,Computer Science,1,85.0
1,2,Computer Science,1,75.0
2,3,Computer Science,2,55.0
3,4,Computer Science,2,45.0
4,5,Computer Science,3,60.0
5,6,Commerce,1,50.0
6,7,Commerce,1,55.0
7,8,Commerce,2,40.0
8,9,Commerce,2,65.0
9,10,Commerce,3,45.0


1. DEPARTMENT-WISE AVERAGE MARKS


,department,average_marks
0,Commerce,51.0
1,Computer Science,64.0
2,Mathematics,85.0
3,Physics,55.0


2. DEPARTMENTS WITH AVERAGE BELOW 60


...

4. DEPARTMENTS WITH AT LEAST 5 STUDENTS


,department,total_students,average_marks,maximum_marks,minimum_marks
0,Commerce,5,51.0,65.0,40.0
1,Computer Science,5,64.0,85.0,45.0
2,Mathematics,5,85.0,95.0,75.0
3,Physics,5,55.0,65.0,45.0


COMPLETE ACADEMIC PERFORMANCE REPORT


,department,total_students,average_marks,maximum_marks,minimum_marks,performance_status
0,Mathematics,5,85.0,95.0,75.0,Satisfactory
1,Computer Science,5,64.0,85.0,45.0,Satisfactory
2,Physics,5,55.0,65.0,45.0,Underperforming
3,Commerce,5,51.0,65.0,40.0,Underperforming


Academic Performance Analytics Completed!


In [5]:
# PROBLEM STATEMENT 5 — Employee Compensation Risk Analysis
# STEP 1: IMPORT LIBRARIES
import sqlite3
import pandas as pd

# STEP 2: CREATE DATABASE CONNECTION
conn = sqlite3.connect("employee_salary.db")
cursor = conn.cursor()

# STEP 3: CREATE TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS employee_salary (
    emp_id INTEGER PRIMARY KEY,
    department TEXT NOT NULL,
    designation TEXT NOT NULL,
    salary REAL NOT NULL CHECK (salary > 0)
)
""")

# STEP 4: INSERT EMPLOYEE DATA
employees = [
    (1, 'HR', 'Manager', 80000),
    (2, 'HR', 'Executive', 35000),
    (3, 'HR', 'Executive', 40000),
    (4, 'HR', 'Analyst', 55000),
    (5, 'IT', 'Manager', 100000),
    (6, 'IT', 'Developer', 75000),
    (7, 'IT', 'Developer', 80000),
    (8, 'IT', 'Developer', 85000),
    (9, 'IT', 'Developer', 90000),
    (10, 'Finance', 'Manager', 90000),
    (11, 'Finance', 'Accountant', 50000),
    (12, 'Finance', 'Accountant', 55000),
    (13, 'Finance', 'Accountant', 60000),
    (14, 'Finance', 'Accountant', 65000),
    (15, 'Sales', 'Manager', 70000),
    (16, 'Sales', 'Sales Executive', 30000),
    (17, 'Sales', 'Sales Executive', 32000),
    (18, 'Sales', 'Sales Executive', 35000),
    (19, 'Sales', 'Sales Executive', 38000)
]

cursor.executemany("""
INSERT OR REPLACE INTO employee_salary
(emp_id, department, designation, salary)
VALUES (?, ?, ?, ?)
""", employees)

conn.commit()

# STEP 5: DISPLAY ALL EMPLOYEE DETAILS
print("ALL EMPLOYEE SALARY DETAILS")

display(pd.read_sql_query("""
SELECT * FROM employee_salary
ORDER BY emp_id
""", conn))

# STEP 6: AVERAGE SALARY PER DEPARTMENT
print("1. AVERAGE SALARY PER DEPARTMENT")

display(pd.read_sql_query("""
SELECT
    department,
    ROUND(AVG(salary), 2) AS average_salary
FROM employee_salary
GROUP BY department
""", conn))

# STEP 7: TOTAL SALARY COST PER DESIGNATION
print("2. TOTAL SALARY COST PER DESIGNATION")

display(pd.read_sql_query("""
SELECT
    designation,
    SUM(salary) AS total_salary_cost
FROM employee_salary
GROUP BY designation
""", conn))

# STEP 8: DEPARTMENTS WITH AVERAGE SALARY > 70000
print("3. DEPARTMENTS WITH AVERAGE SALARY ABOVE 70000")

display(pd.read_sql_query("""
SELECT
    department,
    ROUND(AVG(salary), 2) AS average_salary
FROM employee_salary
GROUP BY department
HAVING AVG(salary) > 70000
""", conn))

# STEP 9: DESIGNATIONS WITH MORE THAN 3 EMPLOYEES
print("4. DESIGNATIONS WITH MORE THAN 3 EMPLOYEES")

display(pd.read_sql_query("""
SELECT
    designation,
    COUNT(emp_id) AS total_employees,
    SUM(salary) AS total_salary_cost,
    ROUND(AVG(salary), 2) AS average_salary
FROM employee_salary
GROUP BY designation
HAVING COUNT(emp_id) > 3
""", conn))

# STEP 10: COMPLETE DEPARTMENT-WISE COST REPORT
print("COMPLETE DEPARTMENT-WISE SALARY REPORT")

display(pd.read_sql_query("""
SELECT
    department,
    COUNT(emp_id) AS total_employees,
    SUM(salary) AS total_salary_cost,
    ROUND(AVG(salary), 2) AS average_salary,
    MAX(salary) AS highest_salary,
    MIN(salary) AS lowest_salary
FROM employee_salary
GROUP BY department
ORDER BY total_salary_cost DESC
""", conn))

# STEP 11: CLOSE DATABASE
conn.close()

print("Employee Compensation Risk Analysis Completed!")



ALL EMPLOYEE SALARY DETAILS


,emp_id,department,designation,salary
0,1,HR,Manager,80000.0
1,2,HR,Executive,35000.0
2,3,HR,Executive,40000.0
3,4,HR,Analyst,55000.0
4,5,IT,Manager,100000.0
5,6,IT,Developer,75000.0
6,7,IT,Developer,80000.0
7,8,IT,Developer,85000.0
8,9,IT,Developer,90000.0
9,10,Finance,Manager,90000.0


1. AVERAGE SALARY PER DEPARTMENT


,department,average_salary
0,Finance,64000.0
1,HR,52500.0
2,IT,86000.0
3,Sales,41000.0


2. TOTAL SALARY COST PER DESIGNATION


...

4. DESIGNATIONS WITH MORE THAN 3 EMPLOYEES


,designation,total_employees,total_salary_cost,average_salary
0,Accountant,4,230000.0,57500.0
1,Developer,4,330000.0,82500.0
2,Manager,4,340000.0,85000.0
3,Sales Executive,4,135000.0,33750.0


COMPLETE DEPARTMENT-WISE SALARY REPORT


,department,total_employees,total_salary_cost,average_salary,highest_salary,lowest_salary
0,IT,5,430000.0,86000.0,100000.0,75000.0
1,Finance,5,320000.0,64000.0,90000.0,50000.0
2,HR,4,210000.0,52500.0,80000.0,35000.0
3,Sales,5,205000.0,41000.0,70000.0,30000.0


Employee Compensation Risk Analysis Completed!


In [6]:
# PROBLEM STATEMENT 6 — Sales Performance & Revenue Control
# STEP 1: IMPORT LIBRARIES
import sqlite3
import pandas as pd

# STEP 2: CREATE DATABASE CONNECTION
conn = sqlite3.connect("sales_performance.db")
cursor = conn.cursor()

# STEP 3: CREATE TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS sales_data (
    order_id INTEGER PRIMARY KEY,
    category TEXT NOT NULL,
    region TEXT NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    price REAL NOT NULL CHECK (price > 0)
)
""")

# STEP 4: INSERT SALES DATA
sales = [
    (1, 'Electronics', 'Chennai', 10, 15000),
    (2, 'Electronics', 'Chennai', 5, 12000),
    (3, 'Electronics', 'Bangalore', 8, 14000),
    (4, 'Electronics', 'Bangalore', 4, 18000),
    (5, 'Furniture', 'Chennai', 6, 10000),
    (6, 'Furniture', 'Madurai', 8, 9000),
    (7, 'Furniture', 'Bangalore', 12, 8500),
    (8, 'Clothing', 'Chennai', 20, 1500),
    (9, 'Clothing', 'Madurai', 30, 1200),
    (10, 'Clothing', 'Bangalore', 25, 1800),
    (11, 'Electronics', 'Madurai', 6, 16000),
    (12, 'Furniture', 'Chennai', 10, 11000),
    (13, 'Electronics', 'Chennai', 3, 20000),
    (14, 'Clothing', 'Madurai', 15, 2000),
    (15, 'Furniture', 'Bangalore', 5, 13000),
    (16, 'Electronics', 'Bangalore', 7, 12500),
    (17, 'Clothing', 'Chennai', 40, 1000),
    (18, 'Furniture', 'Madurai', 4, 15000),
    (19, 'Electronics', 'Chennai', 6, 17000),
    (20, 'Clothing', 'Bangalore', 35, 1600),
    (21, 'Furniture', 'Chennai', 7, 9500),
    (22, 'Electronics', 'Madurai', 5, 19000),
    (23, 'Clothing', 'Chennai', 18, 2200),
    (24, 'Furniture', 'Bangalore', 9, 10500)
]

cursor.executemany("""
INSERT OR REPLACE INTO sales_data
(order_id, category, region, quantity, price)
VALUES (?, ?, ?, ?, ?)
""", sales)

conn.commit()

# STEP 5: DISPLAY ALL SALES DATA
print("ALL SALES DATA")

display(pd.read_sql_query("""
SELECT * FROM sales_data
ORDER BY order_id
""", conn))

# STEP 6: TOTAL REVENUE FOR EACH CATEGORY
print("1. CATEGORY-WISE TOTAL REVENUE")

display(pd.read_sql_query("""
SELECT
    category,
    SUM(quantity * price) AS total_revenue
FROM sales_data
GROUP BY category
ORDER BY total_revenue DESC
""", conn))

# STEP 7: AVERAGE ORDER VALUE PER REGION
print("2. REGION-WISE AVERAGE ORDER VALUE")

display(pd.read_sql_query("""
SELECT
    region,
    ROUND(AVG(quantity * price), 2) AS average_order_value
FROM sales_data
GROUP BY region
""", conn))

# STEP 8: CATEGORIES WITH REVENUE ABOVE 100000
print("3. CATEGORIES WITH TOTAL REVENUE ABOVE 100000")

display(pd.read_sql_query("""
SELECT
    category,
    SUM(quantity * price) AS total_revenue
FROM sales_data
GROUP BY category
HAVING SUM(quantity * price) > 100000
""", conn))

# STEP 9: REGIONS WITH MORE THAN 10 ORDERS
print("4. REGIONS WITH MORE THAN 10 ORDERS")

display(pd.read_sql_query("""
SELECT
    region,
    COUNT(order_id) AS total_orders
FROM sales_data
GROUP BY region
HAVING COUNT(order_id) > 10
""", conn))

# STEP 10: COMPLETE SALES PERFORMANCE REPORT
print("COMPLETE SALES PERFORMANCE REPORT")

display(pd.read_sql_query("""
SELECT
    region,
    COUNT(order_id) AS total_orders,
    SUM(quantity * price) AS total_revenue,
    ROUND(AVG(quantity * price), 2) AS average_order_value
FROM sales_data
GROUP BY region
ORDER BY total_revenue DESC
""", conn))

# STEP 11: CLOSE DATABASE
conn.close()

print("Sales Performance & Revenue Control Completed!")



ALL SALES DATA


,order_id,category,region,quantity,price
0,1,Electronics,Chennai,10,15000.0
1,2,Electronics,Chennai,5,12000.0
2,3,Electronics,Bangalore,8,14000.0
3,4,Electronics,Bangalore,4,18000.0
4,5,Furniture,Chennai,6,10000.0
5,6,Furniture,Madurai,8,9000.0
6,7,Furniture,Bangalore,12,8500.0
7,8,Clothing,Chennai,20,1500.0
8,9,Clothing,Madurai,30,1200.0
9,10,Clothing,Bangalore,25,1800.0


1. CATEGORY-WISE TOTAL REVENUE


,category,total_revenue
0,Electronics,834500.0
1,Furniture,630000.0
2,Clothing,276600.0


2. REGION-WISE AVERAGE ORDER VALUE


...

4. REGIONS WITH MORE THAN 10 ORDERS


,region,total_orders


COMPLETE SALES PERFORMANCE REPORT


,region,total_orders,total_revenue,average_order_value
0,Chennai,10,718100.0,71810.00
1,Bangalore,8,634000.0,79250.00
2,Madurai,6,389000.0,64833.33


Sales Performance & Revenue Control Completed!
